# 04 — Cross-Domain Transfer

Compare beat tracking and pitch estimation on Western vs. Carnatic music across three conditions:

- **A** — pretrained Western model (madmom for beats, CREPE for pitch)
- **B** — model trained from scratch on Saraga
- **C** — Western model fine-tuned on Saraga

The first half of this notebook reads the per-condition result files in `results/` and produces summary plots. The second half shows what the models actually do on a single Saraga track — one figure for beats, one for pitch.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

RESULTS = Path('../results/cross_domain')
long_df = pd.read_csv(RESULTS / 'all_results_long.csv')
summary = pd.read_csv(RESULTS / 'cross_domain_summary.csv')

# Quick check: what (task, condition, domain) cells do we actually have?
long_df.groupby(['task', 'condition', 'domain']).size().unstack(fill_value=0)

## Beat tracking — mean score per (condition, domain)

In [ ]:
beat = summary[summary['task'] == 'beat']
beat_metrics = ['f_measure', 'cemgil', 'continuity', 'downbeat_f_measure']
beat_metrics = [m for m in beat_metrics if m in beat['metric'].unique()]

fig, axes = plt.subplots(1, len(beat_metrics), figsize=(4 * len(beat_metrics), 4))

for ax, metric in zip(axes, beat_metrics):
    sub = beat[beat['metric'] == metric]
    sns.barplot(
        data=sub, x='condition', y='mean', hue='domain',
        order=['A', 'B', 'C'], hue_order=['western', 'carnatic'],
        ax=ax,
    )
    ax.set_title(metric)
    ax.set_xlabel('Condition')
    ax.set_ylabel('Score')
    ax.get_legend().remove()

axes[-1].legend(title='Domain', loc='lower right')
plt.tight_layout()
plt.show()

## Pitch estimation — mean score per (condition, domain)

In [ ]:
pitch = summary[summary['task'] == 'pitch']
pitch_metrics = ['raw_pitch_accuracy', 'overall_accuracy', 'mean_abs_error_cents']
pitch_metrics = [m for m in pitch_metrics if m in pitch['metric'].unique()]

fig, axes = plt.subplots(1, len(pitch_metrics), figsize=(4 * len(pitch_metrics), 4))

for ax, metric in zip(axes, pitch_metrics):
    sub = pitch[pitch['metric'] == metric]
    sns.barplot(
        data=sub, x='condition', y='mean', hue='domain',
        order=['A', 'B', 'C'], hue_order=['western', 'carnatic'],
        ax=ax,
    )
    ax.set_title(metric)
    ax.set_xlabel('Condition')
    ax.set_ylabel('cents' if 'error' in metric else 'Score')
    ax.get_legend().remove()

axes[-1].legend(title='Domain', loc='lower right')
plt.tight_layout()
plt.show()

## Transfer gap (Western − Carnatic)

A positive bar means the model scores better on Western than on Carnatic for that condition. Condition A should show the largest gap; B and C should shrink it.

In [ ]:
focus = {'beat': 'f_measure', 'pitch': 'raw_pitch_accuracy'}

rows = []
for task, metric in focus.items():
    piv = (
        summary[(summary['task'] == task) & (summary['metric'] == metric)]
        .pivot(index='condition', columns='domain', values='mean')
    )
    if {'western', 'carnatic'}.issubset(piv.columns):
        for cond, r in piv.iterrows():
            rows.append({
                'task': task,
                'condition': cond,
                'delta': r['western'] - r['carnatic'],
            })
gap = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=gap, x='condition', y='delta', hue='task', order=['A', 'B', 'C'], ax=ax)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Western score − Carnatic score')
ax.set_title('Transfer gap by condition')
plt.tight_layout()
plt.show()

gap

## Per-track distributions

Box plots of the per-track scores, so we can tell whether the mean differences are systematic or driven by a few outliers.

In [ ]:
per_track = long_df[long_df['track_id'] != '__aggregate__']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (task, metric, title) in zip(axes, [
    ('beat', 'f_measure', 'Beat F-measure per track'),
    ('pitch', 'raw_pitch_accuracy', 'Pitch RPA per track'),
]):
    sub = per_track[(per_track['task'] == task) & (per_track['metric'] == metric)]
    sns.boxplot(
        data=sub, x='condition', y='value', hue='domain',
        order=['A', 'B', 'C'], hue_order=['western', 'carnatic'], ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Condition')
    ax.set_ylabel(metric)

plt.tight_layout()
plt.show()

## Demo plots on one Saraga track

The summary plots above show averages. Here we pick a single Carnatic track from Saraga and show what the models actually produce — first the predicted beat positions over the waveform, then the predicted pitch contour against the reference F0.

> **Note.** These cells assume Saraga is downloaded at the path given in `config/config.yaml`. If it isn't, the cell will raise a clear error — download Saraga via `python scripts/download_data.py` first.

In [ ]:
from src.preprocessing.common import load_config
from src.preprocessing.saraga import get_saraga_split

config = load_config('../config/config.yaml')
saraga_path = config['datasets']['saraga']['path']

# Load a handful of tracks and pick the first one that has *both* beat
# annotations (sama) and pitch annotations, so we can show both demos
# on the same piece.
tracks = get_saraga_split(saraga_path, split='test', max_tracks=10)

demo_track = next(
    (t for t in tracks if t.beat_times is not None and t.f0_freqs is not None),
    None,
)
if demo_track is None:
    raise RuntimeError('No Saraga track in the test split has both sama and pitch annotations.')

print(f'Demo track: {demo_track.track_id}')
print(f'  duration : {len(demo_track.audio) / demo_track.sr:.1f} s')
print(f'  raga     : {demo_track.metadata.get("raga")}')
print(f'  tala     : {demo_track.metadata.get("tala")}')

### Beat tracking demo — madmom on a Saraga track

In [ ]:
from src.rhythm.beat_tracking import run_beat_tracking

beat_pred = run_beat_tracking(demo_track, backend='madmom')

# Plot a short window so the beats are readable.
window = 12.0
t_end = min(window, len(demo_track.audio) / demo_track.sr)
n_samples = int(t_end * demo_track.sr)
times = np.arange(n_samples) / demo_track.sr

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(times, demo_track.audio[:n_samples], color='gray', lw=0.6, alpha=0.7)

for b in beat_pred.beat_times[beat_pred.beat_times <= t_end]:
    ax.axvline(b, color='tab:red', lw=1.3)

ref = demo_track.beat_times
for b in ref[ref <= t_end]:
    ax.axvline(b, color='tab:green', lw=1.3, ls='--')

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color='gray', lw=1, label='waveform'),
    Line2D([0], [0], color='tab:red', lw=1.5,
           label=f'predicted (madmom, {beat_pred.bpm:.0f} BPM)'),
    Line2D([0], [0], color='tab:green', lw=1.5, ls='--', label='reference (sama)'),
])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f'Beat tracking on {demo_track.track_id} (first {t_end:.0f}s)')
plt.tight_layout()
plt.show()

### Pitch estimation demo — CREPE on the same Saraga track

In [ ]:
from src.pitch.estimation import run_pitch_estimation

pitch_pred = run_pitch_estimation(demo_track, method='crepe')

# Drop unvoiced frames so they don't draw spurious zeros on a log axis.
pred_freq = pitch_pred.frequencies.astype(float)
pred_freq = np.where(pred_freq > 0, pred_freq, np.nan)

ref_freq = demo_track.f0_freqs.astype(float)
ref_freq = np.where(ref_freq > 0, ref_freq, np.nan)

t_end = min(20.0, len(demo_track.audio) / demo_track.sr)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(demo_track.f0_times, ref_freq, color='tab:green', lw=1.3, ls='--', label='reference F0')
ax.plot(pitch_pred.times, pred_freq, color='tab:red', lw=1.3, label='predicted F0 (CREPE)')

ax.set_yscale('log')
ax.set_xlim(0, t_end)
ax.set_ylim(80, 1000)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz, log scale)')
ax.set_title(f'Pitch estimation on {demo_track.track_id} (first {t_end:.0f}s)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Final 3 × 2 tables

In [ ]:
from src.cross_domain.experiments import pivot_table

for task, metric in [
    ('beat', 'f_measure'),
    ('beat', 'downbeat_f_measure'),
    ('pitch', 'raw_pitch_accuracy'),
    ('pitch', 'mean_abs_error_cents'),
]:
    table = pivot_table(summary, metric=metric, task=task)
    if table.empty:
        continue
    print(f'\n--- {task} | {metric} ---')
    print(table.round(3).to_string())